# Tool Binding and Multimodal Stimuli

Two features that don't fit the survey/RM tutorials: giving the model
**tools** it can call (`card.tools`), and giving it **multimodal** stimuli —
images, audio, files, and scraped web text — instead of plain strings (see
`psychscanner.datasets.prompts.multimodal`).

This notebook covers:
1. `card.tools` — binding LangChain tools to the model for a whole run
2. Per-trial `"tools"` subsetting from the task JSON
3. The four multimodal content-block builders: `image_block`, `audio_block`,
   `file_block`, `website_block`
4. The same multimodal stimulus run under all three `chain_type`/`memory`
   pipeline combinations the engine supports (see `docs/guides/cognitive_tasks.md`)
5. Tools + multimodal + CSV export, combined in one `ExpCard` run

Tool-calling needs a model that actually supports `bind_tools`; `mock-llm`
does not, so sections 1-2 use the local `llama3.2:3b` Ollama model (pulled
for this tutorial). `llama3.2:3b` is text-only, so sections 4-5, which are
about the multimodal *pipeline mechanics* rather than model image quality,
use `mock-llm` — swap in any vision-capable provider (`gpt-4o`, `claude-*`,
a local vision model) for real image understanding; nothing else changes.

In [1]:
from pathlib import Path

from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

import psychscanner as psy
from psychscanner.datasets.prompts.multimodal import (
    image_block, audio_block, file_block, website_block, resolve_path_block,
)
from psychscanner.task_runner import TaskRunner

print("PsychScanner successfully imported!")

PsychScanner successfully imported!


In [2]:
import base64

RUN_DIR = Path.cwd() / "_tool_multimodal_tutorial_run"
RUN_DIR.mkdir(exist_ok=True)

# A tiny 1x1 PNG, just so image_block() has a real file to read.
png_bytes = base64.b16decode(
    "89504E470D0A1A0A0000000D49484452000000010000000108020000009077"
    "53DE0000000C4944415478DA6360000002000155A1F6B50000000049454E44AE426082"
)
(RUN_DIR / "pixel.png").write_bytes(png_bytes)

# A minimal (silent) WAV header — enough bytes to demo the block builder, no real audio needed.
(RUN_DIR / "tone.wav").write_bytes(bytes.fromhex(
    "524946462400000057415645666d74201000000001000100401f0000401f00000100080064617461000000"
))

(RUN_DIR / "notes.txt").write_text("Participant reported feeling relaxed during the scan.")
print("media files ready:", [p.name for p in RUN_DIR.iterdir()])

media files ready: ['notes.txt', 'pixel.png', 'tone.wav']


## 1. `card.tools` — binding tools for a whole run

`card.tools` is a card-level hyperparameter (like `parameters`) — every
persona and trial in one `ExpCard` shares the same tool pool. The built-in
agent (`memories.single_turn_convo`) calls `model.bind_tools(...)` so the
model *can* request a tool, but — unlike `psychscanner.agents.make_react_agent`
in the next notebook — it never actually executes the call and loops; it just
returns whatever `AIMessage` (with or without `tool_calls`) the model produced
for that one turn. That's enough to see the model decide to reach for a tool,
which is what this section shows.

In [3]:
@tool
def image_zoom(region: str) -> str:
    """Return a zoomed-in crop of the display for the named region (e.g. 'top-left')."""
    return f"[zoomed crop of {region}]"


task = {
    "tasktype": "visual_search", "taskname": "feature_search",
    "instructions": {"definition": [
        "Decide whether the target is present. Use the image_zoom tool if you need a closer look at a region first."
    ]},
    "contexts": ["Feature search (pop-out)"], "contexts_id": ["feat"],
    "context_present": False, "chain_type": "item",
    "parser": "0",
    "items": {"feat": [
        {"trcode": "feat_1", "stimulus": "A red circle target sits among blue circle distractors. Is the target present? yes/no."},
    ]},
}

card_in = psy.ExpCardInit()
card_in.proj_dir = RUN_DIR
card_in.projectname = "tool_binding_demo"
card_in.model = "llama3.2:3b"
card_in.family = "ollama"
card_in.parameters = {"temperature": 0}
card_in.task_file = task
card_in.tools = [image_zoom]
card_in.cogtype = "no"
card_in.nsim = 1
card_in.memory = "SingleTurn"
card_in.chain_type = "item"

scanner = psy.ScannerModel(expcard=psy.ExpCard(card_in))
results = scanner.run()
resp = results[0][0]["pred_resp"]
print("content:", resp.content)
print("tool_calls:", getattr(resp, "tool_calls", None))

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run/tool_binding_demo/feature_search/ollama_llama3.2:3b_SingleTurn


----<>----


--<api key>-- warning: OLLAMA_API_KEY not set; proceeding without explicit api_key for family 'ollama'


...

1it [00:04,  4.32s/it]


2026-07-06 11:42:23.577 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:42:23.580 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


content: 
tool_calls: [{'name': 'image_zoom', 'args': {'region': 'target'}, 'id': 'afad51a1-90f2-4532-943f-ee552e922805', 'type': 'tool_call'}]


## 2. Per-trial `"tools"` subsetting

A trial's `"tools"` key selects a subset of `card.tools` by name: absent
falls back to the full pool, `[]` opts a trial out entirely. Demonstrated
directly through `TaskRunner` (no `ExpCard` needed) against the same model,
one trial per case.

In [4]:
@tool
def web_search(query: str) -> str:
    """Search the web for a query."""
    return f"[results for {query}]"


from psychscanner.memories import llm_chat_model
from psychscanner.scanner_models.agent_config import AgentConfig
from psychscanner.memories.single_turn_convo import single_turn_convo_node
from psychscanner.memories.base.base_agent import AgentInitializer

agent_cfg = AgentConfig(
    modelname="llama3.2:3b", familyname="ollama", parameters={"temperature": 0},
    modelobject=llm_chat_model(model="llama3.2:3b", family="ollama", parameters={"temperature": 0}),
    memory_type="SingleTurn", memory_k=-1, summary_k=0, chain_type="item",
    system_msg=None, parser=None, parser_raw=False, parser_config={},
    tools=[image_zoom, web_search],
)
agent = AgentInitializer(agent_cfg=agent_cfg)
agent.ai_app = single_turn_convo_node(agent_cfg)

tasktrials = {"trials": [
    {"trcode": "t1", "stimulus": HumanMessage(content="Zoom the top-left region and tell me what tool you used."),
     "tasktype": "x", "parser": None, "fb": False, "tools": ["image_zoom"]},
    {"trcode": "t2", "stimulus": HumanMessage(content="What's 2+2? Do not use any tool."),
     "tasktype": "x", "parser": None, "fb": False, "tools": []},
    {"trcode": "t3", "stimulus": HumanMessage(content="Search the web for psychscanner and summarize."),
     "tasktype": "x", "parser": None, "fb": False},
]}
runner = TaskRunner(
    scanning_agent=agent, trace_cfg={"trial": "tut-", "task": "tut-task"},
    system_message="You are a participant with access to tools.",
    tasktrials=tasktrials, chain_type="item", hmsg="stimulus",
)
for r in runner.execute():
    print(r["trcode"], "-> tool_calls:", getattr(r["pred_resp"], "tool_calls", None), "| content:", r["pred_resp"].content[:80])

--<api key>-- warning: OLLAMA_API_KEY not set; proceeding without explicit api_key for family 'ollama'


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model='llama3.2:3b' temperature=0.0


----<>---- task running


t1 -> tool_calls: [{'name': 'image_zoom', 'args': {'region': 'top-left'}, 'id': 'fea7f8a5-97c5-49a6-9b2b-90b16c4ae982', 'type': 'tool_call'}] | content: 
t2 -> tool_calls: [] | content: A simple math question!

2 + 2 = 4
t3 -> tool_calls: [{'name': 'web_search', 'args': {'query': 'psychscanner summary'}, 'id': 'a5e9dc1a-9161-44be-a537-334430b12a59', 'type': 'tool_call'}] | content: 


## 3. Multimodal content blocks

`image_block` / `audio_block` / `file_block` turn a local path or URL into a
standard LangChain content block; `website_block` fetches a page and returns
its visible text (needs the `multimodal` extra: `httpx` + `beautifulsoup4`).
`resolve_path_block` lets hand-authored task JSON reference `{"type":
"image", "path": "..."}` instead of calling `image_block()` in Python.

In [5]:
def _preview(block):
    return {k: (v[:20] + "...") if k == "base64" else v for k, v in block.items()}

print("image_block:", _preview(image_block(RUN_DIR / "pixel.png")))
print("audio_block:", _preview(audio_block(RUN_DIR / "tone.wav")))
print("file_block: ", _preview(file_block(RUN_DIR / "notes.txt")))
print("resolve_path_block:", _preview(resolve_path_block({"type": "image", "path": str(RUN_DIR / "pixel.png")})))

web_block = website_block("https://example.com")
print("website_block:", {**web_block, "text": web_block["text"][:60] + "..."})

image_block: {'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhE...', 'mime_type': 'image/png'}
audio_block: {'type': 'audio', 'base64': 'UklGRiQAAABXQVZFZm10...', 'mime_type': 'audio/x-wav'}
file_block:  {'type': 'file', 'base64': 'UGFydGljaXBhbnQgcmVw...', 'mime_type': 'text/plain'}
resolve_path_block: {'type': 'image', 'base64': 'iVBORw0KGgoAAAANSUhE...', 'mime_type': 'image/png'}


website_block: {'type': 'text-plain', 'text': 'Example Domain\nExample Domain\nThis domain is for use in docu...', 'mime_type': 'text/plain'}


## 4. The same multimodal stimulus, all three pipeline setups

From `docs/guides/cognitive_tasks.md`: a multimodal stimulus works under any
`memory`/`chain_type` combination — `thread_id` (which decides how much
context carries over) is orthogonal to what's in the stimulus.

| Configuration | `memory` | `chain_type` | Use for |
|---|---|---|---|
| Single trial   | `SingleTurn` | `item`  | Feature/pop-out search — each display independent |
| Trial-chain    | `Convo`      | `trial` | Serial attention scan within one display |
| Episodic chain | `Convo`      | `task`  | Sustained-attention / oddball vigilance block |

All three use `mock-llm` (`ExpCardInit`'s default `family`/`model`) so this
cell runs fast and hardware-independent — the mock model just echoes the
first few characters of what it received, which is enough to show the
content blocks arrived intact. Point `card_in.model`/`card_in.family` at any
vision-capable provider for real image understanding; nothing else changes.

In [6]:
def search_trial(trcode, question):
    return {"trcode": trcode, "stimulus": [image_block(RUN_DIR / "pixel.png"), {"type": "text", "text": question}]}


def run_card(memory, chain_type, items, projectname, **extra):
    task = {
        "tasktype": "visual_search", "taskname": projectname,
        "instructions": {"definition": ["Look at the image and answer the question."]},
        "contexts": ["demo"], "contexts_id": ["feat"], "context_present": False,
        "chain_type": chain_type, "parser": "0", "items": {"feat": items},
    }
    card_in = psy.ExpCardInit()
    card_in.proj_dir, card_in.projectname = RUN_DIR, projectname
    card_in.task_file = task
    card_in.cogtype, card_in.nsim = "no", 1
    card_in.memory, card_in.chain_type = memory, chain_type
    for k, v in extra.items():
        setattr(card_in, k, v)
    scanner = psy.ScannerModel(expcard=psy.ExpCard(card_in))
    return scanner.run()

In [7]:
def block_types(content):
    if not isinstance(content, list):
        return content
    return [b.get("type") if isinstance(b, dict) else "text" for b in content]


# Single trial: SingleTurn + item — each display independent
single_results = run_card("SingleTurn", "item", [
    search_trial("feat_1", "Is the pixel red, green, or blue? One word."),
], "single_trial_demo")
for trial in single_results[0]:
    print(trial["trcode"], "-> block types echoed back:", block_types(trial["pred_resp"].content))

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run/single_trial_demo/single_trial_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

1it [00:00, 269.97it/s]


2026-07-06 11:42:35.624 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:42:35.625 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


feat_1 -> block types echoed back: ['image', 'text']


In [8]:
# Trial-chain: Convo + trial — sub-stimuli share one trcode, so they land on
# the same LangGraph thread and accumulate memory only within that trial.
chain_results = run_card("Convo", "trial", [
    search_trial("feat_1", "Note the color, then say NEXT."),
    search_trial("feat_1", "Now, based on what you noted, what color was it?"),
], "trial_chain_demo")
for trial in chain_results[0]:
    print(trial["trcode"], "-> block types echoed back:", block_types(trial["pred_resp"].content))

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run/trial_chain_demo/trial_chain_demo/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

2it [00:00, 135.80it/s]


2026-07-06 11:42:35.673 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:42:35.674 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


feat_1 -> block types echoed back: ['image', 'text']
feat_1 -> block types echoed back: ['image', 'text']


In [9]:
# Episodic chain: Convo + task — the whole block is one conversation;
# tunnel_status="1" also checkpoints progress after each system message.
vig_results = run_card("Convo", "task", [
    search_trial("feat_1", "Oddball present? yes/no."),
    search_trial("feat_2", "Oddball present? yes/no."),
], "episodic_chain_demo", tunnel_status="1")
for trial in vig_results[0]:
    print(trial["trcode"], "-> block types echoed back:", block_types(trial["pred_resp"].content))

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run/episodic_chain_demo/episodic_chain_demo/mock-llm_mock-chat-model_Convo


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

2it [00:00, 166.95it/s]


2026-07-06 11:42:35.719 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:42:35.721 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


feat_1 -> block types echoed back: ['image', 'text']
feat_2 -> block types echoed back: ['image', 'text']


## 5. Putting it together: tools + multimodal + CSV export

A card can combine `tools` and multimodal stimuli in one `ExpCard`. This
trial opts out of the card's tools via `"tools": []` (see section 2) so it
can still run against `mock-llm` deterministically — swap in a tool-calling
model (as in sections 1-2) to see a trial actually reach for `image_zoom`.
Exported to CSV exactly like any other run: `base64` payloads are stripped
from the CSV; `.psyscan` checkpoints keep the media, content-addressed under
`<data_root_dir>/media/`.

In [10]:
combined_task = {
    "tasktype": "visual_search", "taskname": "combined_demo",
    "instructions": {"definition": ["Look at the image and answer."]},
    "contexts": ["demo"], "contexts_id": ["feat"], "context_present": False,
    "chain_type": "item", "parser": "0",
    "items": {"feat": [{**search_trial("feat_1", "Is the target present? yes/no."), "tools": []}]},
}
card_in = psy.ExpCardInit()
card_in.proj_dir, card_in.projectname = RUN_DIR, "combined_demo"
card_in.task_file = combined_task
card_in.tools = [image_zoom]
card_in.cogtype, card_in.nsim = "no", 1
card_in.memory, card_in.chain_type = "SingleTurn", "item"

scanner = psy.ScannerModel(expcard=psy.ExpCard(card_in))
scanner.run()

from psychscanner import to_csv
df = to_csv(scanner, path=RUN_DIR / "combined_demo.csv")
df.select(["trcode", "pred_resp_raw"])

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run/combined_demo/combined_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...


2026-07-06 11:42:35.768 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 0


2026-07-06 11:42:35.771 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


Saved 1 rows → /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_tool_multimodal_tutorial_run/combined_demo.csv


trcode,pred_resp_raw
str,str
"""feat_1""","""[{'type': 'image', 'base64': '…"


## Recap

- `card.tools` binds LangChain tools for a whole run; the built-in agent
  binds them but doesn't execute/loop — see `psychscanner.agents.make_react_agent`
  (next notebook) for a real tool-execution loop.
- A trial's `"tools"` key subsets `card.tools` per trial (`None` = full pool,
  `[]` = opt out).
- `image_block`/`audio_block`/`file_block`/`website_block` build standard
  content blocks from local files or URLs; `resolve_path_block` resolves
  JSON-authored `{"path": ...}` blocks.
- The same multimodal stimulus works under every `memory`/`chain_type`
  combination — they control context carry-over, not what's in the stimulus.
- Tools and multimodal stimuli combine freely in one `ExpCard`, and export to
  CSV like any other run.

---
## Further reading

Advanced applications of tool binding and multimodal stimulus handling:

1. **["Tool Learning with Large Language Models: A Survey"](https://arxiv.org/abs/2405.17935)** (Qu et al., 2024) — surveys task planning, tool selection, and calling, the same stages exercised by `card.tools` and per-trial `"tools"` subsetting above, at the scale of dozens of tools instead of one.
2. **["VisualAgentBench: Towards Large Multimodal Models as Visual Foundation Agents"](https://arxiv.org/abs/2408.06327)** (2024) — benchmarks multimodal models as agents grounded in real visual environments, the natural next step once a vision-capable provider is swapped in for `mock-llm` here.